# EEG analysis in Google Colab

Run the cells from top to bottom with **Shift+Enter**. The first cell installs the
toolkit (about 20 seconds); you do not need to restart the runtime.

Source: https://github.com/rsd21/eeg-analysis

In [ ]:
!pip install -q git+https://github.com/rsd21/eeg-analysis.git

import eegproc as ep
print("eegproc", ep.__version__, "is ready")

## 1. Load a recording

Leave `UPLOAD = False` to try a simulated 2-minute, 32-channel recording with eye blinks.
Set `UPLOAD = True` to upload your own file: EDF, BDF, BrainVision (select the `.vhdr`,
`.vmrk` and `.eeg` files together), CSV/TXT, MATLAB `.mat`, EEGLAB `.set`, HDF5 or XDF.

In [ ]:
UPLOAD = False

if UPLOAD:
    from google.colab import files
    uploaded = list(files.upload())
    path = next((f for f in uploaded if f.lower().endswith((".vhdr", ".set"))), uploaded[0])
    raw = ep.read_raw(path)
    raw.set_montage("standard_1010")      # electrode positions for 10-20 / 10-10 channel names
else:
    raw = ep.simulate_eeg(duration=120, montage="32", eog=True, seed=42)

print(raw.describe())

## 2. Which channels are performing well?

Every channel gets a 0-100 score from 12 metrics (flat, noise, line noise, drift, artifacts, ...).

In [ ]:
quality = raw.assess_quality()
print(quality.summary())
quality.plot();

## 3. Detect eye blinks

Blinks are found on the EOG channel, or on Fp1/Fp2 when the recording has no EOG.

In [ ]:
from eegproc.preprocessing import find_blinks

raw = raw.filter(1.0, 40.0)
blinks = find_blinks(raw)
minutes = raw.n_times / raw.sfreq / 60
print(f"{len(blinks)} blinks in {minutes:.1f} min -> {len(blinks) / minutes:.1f} blinks/min")

## 4. Remove the blinks with ICA

In [ ]:
ica = ep.ICA(n_components=0.99, random_state=0).fit(raw)
eye, scores = ica.find_bads_eog(raw)
ica.exclude = eye
clean = ica.apply(raw)
print(f"Removed {len(eye)} eye component(s) out of {ica.n_components_}:",
      ", ".join(f"IC{i} (r = {scores[i]:.2f})" for i in eye))
ica.plot_components();

### Before vs after: the average blink on a frontal channel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from eegproc.viz import CATEGORICAL, style

ch = next(c for c in ("Fp1", "Fp2", "AF3", "F3", raw.pick_names("eeg")[0]) if c in raw.ch_names)
half = int(0.5 * raw.sfreq)
peaks = ((blinks.onset + blinks.duration / 2) * raw.sfreq).astype(int)
peaks = peaks[(peaks >= half) & (peaks < raw.n_times - half)]
lag = (np.arange(2 * half) - half) / raw.sfreq * 1000
before = np.mean([raw[ch][p - half:p + half] for p in peaks], axis=0)
after = np.mean([clean[ch][p - half:p + half] for p in peaks], axis=0)

with style():
    fig, ax = plt.subplots(figsize=(8, 3.5), layout="constrained")
    ax.plot(lag, before, color=CATEGORICAL[0], lw=2, label="before ICA")
    ax.plot(lag, after, color=CATEGORICAL[1], lw=2, label="after ICA")
    ax.set(xlabel="Time from blink peak (ms)", ylabel="µV")
    ax.set_title(f"Average of {len(peaks)} blinks at {ch}", loc="left")
    ax.legend()
print(f"Blink artifact at {ch}: {np.ptp(before):.0f} µV -> {np.ptp(after):.0f} µV peak-to-peak")

## 5. Save the cleaned data and a full HTML report

In [ ]:
clean.save("clean.edf")
ep.generate_report(clean, "report.html", title="EEG report")

try:
    from google.colab import files
    files.download("report.html")
    files.download("clean.edf")
except ImportError:
    print("Saved report.html and clean.edf next to this notebook")